In [ ]:
#AI was used to help generate this code. Code was tested and verified.
#Model used to generate code: qwen-coder:30b

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import sys

%matplotlib inline

In [ ]:
def load_transactions(file_path):

    try:
        #Use pandas to read csv file in absolute path. See last block. 
        df = pd.read_csv(file_path)
        
        #Checking all columns are in CSV file
        required_columns = ['Type', 'Account', 'Item', 'Category', 'Amount', 'Date']
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"CSV must contain columns: {required_columns}")
        
        #Convert Date column to datetime
        df['Date'] = pd.to_datetime(df['Date'])
        
        #Sort by date. Reset index and get rid of old index
        df = df.sort_values('Date').reset_index(drop=True)
        
        return df
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        raise  

def create_mirror_transfers(df):
    #Create mirror transfer transactions by swapping Account and Item field values
    # Filter for transfer transactions
    transfer_df = df[df['Type'] == 'Transfer'].copy()
    
    #Swap the values in Account and Item columns.
    temp = transfer_df['Account'].copy()
    transfer_df['Account'] = transfer_df['Item']
    transfer_df['Item'] = temp
    
    #Make the amounts negative
    transfer_df['Amount'] = -transfer_df['Amount']

    #Reset index
    transfer_df = transfer_df.reset_index(drop=True)
    
    return transfer_df

def calculate_account_balances(df, transfers_df=None):
    #Calculate running balance for each account
    #Create a copy to avoid modifying original data
    df_copy = df.copy()
    
    #If transfers_df is provided, merge it with the original df
    if transfers_df is not None:
        #Combine original transactions with mirror transfers
        df_copy = pd.concat([df_copy, transfers_df], ignore_index=True)
        #Sort by date again after merging
        df_copy = df_copy.sort_values('Date').reset_index(drop=True)
    
    #Create empty dictionary to track balances
    account_balances = {}
    
    #Process each transaction
    for idx, row in df_copy.iterrows():
        transaction_type = row['Type']
        account = row['Account']
        amount = row['Amount']
        
        if account not in account_balances:
            account_balances[account] = 0

        if transaction_type == "Withdrawal":
            account_balances[account] -= amount
        elif transaction_type == "Deposit":
            account_balances[account] += amount
        elif transaction_type == "Credit Payment":
            account_balances['Account 3'] -= amount
            account_balances['Credit Charge'] += amount
        elif transaction_type == "Transfer":
            account_balances[account] -= amount
        df_copy.loc[idx, 'Balance'] = account_balances[account]
    
    return df_copy

def print_ending_balances(df_with_balances):
    #Get and print the ending balance for each account"
    #Get the ending balance for each account 
    ending_balances = df_with_balances.groupby('Account')['Balance'].last()
    
    print("\n--- Ending Balances ---")
    for account, balance in ending_balances.items():
        #Don't Show Credit Payment
        if account != 'Credit Payment':
            print(f"{account}: ${balance:.2f}")
    
    #Print total balance across all accounts (excluding Credit Payment)
    total_balance = ending_balances.sum()
    print(f"\nTotal Balance: ${total_balance:.2f}")

def plot_account_balances(df):
    #Ensure Date column is datetime
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    #Filter out Credit Payment transactions for plotting
    df_plot = df[df['Type'] != 'Credit Payment']
    
    #Create a pivot table for plotting
    pivot_df = df_plot.pivot_table(index='Date', columns='Account', values='Balance', aggfunc='last')
    
    #Create the plot
    plt.figure(figsize=(12, 8))
    
    #Plot each account
    for account in pivot_df.columns:
        plt.scatter(pivot_df.index, pivot_df[account], marker='o', label=account)
    
    plt.title('Account Balances Over Time')
    plt.xlabel('Date')
    plt.ylabel('Balance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Show the plot
    plt.show()


In [ ]:
file_path = r"C:\absolute\path\transactions_matplotlib.csv"  # Replace with actual path

# Load transactions
df = load_transactions(file_path)
transfers_df = create_mirror_transfers(df)

# Calculate balances
df_with_balances = calculate_account_balances(df, transfers_df)

# Print ending balances
print_ending_balances(df_with_balances)

# Plot account balances
plot_account_balances(df_with_balances)
